# Clean Dataset - Cancer Prediction

Notebook này làm sạch dataset **cancer patient data sets.xlsx** theo đúng hướng xử lý đã trình bày trong báo cáo Big Data: dữ liệu gốc từ Excel được chuẩn hóa để đưa sang CSV, phục vụ HDFS/MapReduce, MongoDB và bước huấn luyện mô hình Machine Learning.

## 1. Mục tiêu xử lý dữ liệu

Theo báo cáo, dataset có 1.000 bệnh nhân và 25 cột. Trong đó:

- `Patient Id` là mã định danh bệnh nhân, dùng để truy vết và lưu vào MongoDB.
- `Level` là nhãn đầu ra gồm `Low`, `Medium`, `High`.
- 23 cột còn lại là đặc trưng đầu vào cho mô hình Random Forest/XGBoost.
- Một số thống kê quan trọng cho Big Data/MapReduce gồm phân bố `Level`, phân bố theo nhóm tuổi và quan hệ giữa `Smoking` với `Level`.

Vì dữ liệu gốc đã khá sạch, phần xử lý quan trọng nhất không phải là xóa nhiều dòng, mà là **chuẩn hóa schema, kiểm định miền giá trị, tạo biến phục vụ thống kê batch và tạo file ML-ready**.

## 2. Khai báo thư viện

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np



pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

## 3. Đọc dữ liệu Excel

Đường dẫn bên dưới trỏ trực tiếp tới file dataset trong project Big Data. Khi chuyển máy, chỉ cần đổi lại `DATA_PATH`.

In [3]:
DATA_PATH = Path(r'D:\\1BigDataproject1\\cancer patient data sets.xlsx')
df_raw = pd.read_excel(DATA_PATH)

print('Shape:', df_raw.shape)
display(df_raw.head())

Shape: (1000, 25)


,Patient Id,Age,Gender,Air Pollution,Alcohol use,Dust Allergy,OccuPational Hazards,Genetic Risk,chronic Lung Disease,Balanced Diet,Obesity,Smoking,Passive Smoker,Chest Pain,Coughing of Blood,Fatigue,Weight Loss,Shortness of Breath,Wheezing,Swallowing Difficulty,Clubbing of Finger Nails,Frequent Cold,Dry Cough,Snoring,Level
0,P1,33,1,2,4,5,4,3,2,2,4,3,2,2,4,3,4,2,2,3,1,2,3,4,Low
1,P10,17,1,3,1,5,3,4,2,2,2,2,4,2,3,1,3,7,8,6,2,1,7,2,Medium
2,P100,35,1,4,5,6,5,5,4,6,7,2,3,4,8,8,7,9,2,1,4,6,7,2,High
3,P1000,37,1,7,7,7,7,6,7,7,7,7,7,7,8,4,2,3,1,4,5,6,7,5,High
4,P101,46,1,6,8,7,7,7,6,7,7,8,7,7,9,3,2,4,1,4,2,4,2,3,High


## 4. Ý nghĩa các nhóm cột

Dataset này khác Telco Churn ở chỗ hầu hết biến đầu vào đã là số. Do đó ta không cần One-Hot Encoding nhiều cột phân loại. Các cột quan trọng được chia như sau:

- **Định danh:** `Patient Id`.
- **Thông tin cơ bản:** `Age`, `Gender`.
- **Yếu tố môi trường/lối sống:** `Air Pollution`, `Alcohol use`, `Dust Allergy`, `OccuPational Hazards`, `Smoking`, `Passive Smoker`.
- **Yếu tố nền và sức khỏe:** `Genetic Risk`, `chronic Lung Disease`, `Balanced Diet`, `Obesity`.
- **Triệu chứng:** `Chest Pain`, `Coughing of Blood`, `Fatigue`, `Weight Loss`, `Shortness of Breath`, `Wheezing`, `Swallowing Difficulty`, `Clubbing of Finger Nails`, `Frequent Cold`, `Dry Cough`, `Snoring`.
- **Nhãn dự đoán:** `Level`.

In [4]:
column_groups = {
    'id': ['Patient Id'],
    'basic_info': ['Age', 'Gender'],
    'environment_lifestyle': ['Air Pollution', 'Alcohol use', 'Dust Allergy', 'OccuPational Hazards', 'Smoking', 'Passive Smoker'],
    'health_background': ['Genetic Risk', 'chronic Lung Disease', 'Balanced Diet', 'Obesity'],
    'symptoms': ['Chest Pain', 'Coughing of Blood', 'Fatigue', 'Weight Loss', 'Shortness of Breath', 'Wheezing', 'Swallowing Difficulty', 'Clubbing of Finger Nails', 'Frequent Cold', 'Dry Cough', 'Snoring'],
    'target': ['Level'],
}

for group, columns in column_groups.items():
    print(f'{group}: {len(columns)} cột')
    print(columns)

id: 1 cột
['Patient Id']
basic_info: 2 cột
['Age', 'Gender']
environment_lifestyle: 6 cột
['Air Pollution', 'Alcohol use', 'Dust Allergy', 'OccuPational Hazards', 'Smoking', 'Passive Smoker']
health_background: 4 cột
['Genetic Risk', 'chronic Lung Disease', 'Balanced Diet', 'Obesity']
symptoms: 11 cột
['Chest Pain', 'Coughing of Blood', 'Fatigue', 'Weight Loss', 'Shortness of Breath', 'Wheezing', 'Swallowing Difficulty', 'Clubbing of Finger Nails', 'Frequent Cold', 'Dry Cough', 'Snoring']
target: 1 cột
['Level']


## 5. Kiểm tra tổng quan dữ liệu

Bước này tương tự notebook Machine Learning mẫu: kiểm tra số dòng/cột, kiểu dữ liệu, missing value và dữ liệu trùng.

In [5]:
print('Số dòng, số cột:', df_raw.shape)
print('\nKiểu dữ liệu:')
display(df_raw.dtypes.to_frame('dtype'))

print('\nSố missing value theo cột:')
display(df_raw.isna().sum().to_frame('missing_count'))

print('Số dòng trùng toàn bộ:', df_raw.duplicated().sum())
print('Số Patient Id bị trùng:', df_raw['Patient Id'].duplicated().sum())

Số dòng, số cột: (1000, 25)

Kiểu dữ liệu:


,dtype
Patient Id,object
Age,int64
Gender,int64
Air Pollution,int64
Alcohol use,int64
Dust Allergy,int64
OccuPational Hazards,int64
Genetic Risk,int64
chronic Lung Disease,int64
Balanced Diet,int64



Số missing value theo cột:


,missing_count
Patient Id,0
Age,0
Gender,0
Air Pollution,0
Alcohol use,0
Dust Allergy,0
OccuPational Hazards,0
Genetic Risk,0
chronic Lung Disease,0
Balanced Diet,0


Số dòng trùng toàn bộ: 0
Số Patient Id bị trùng: 0


### Nhận xét

Dataset gốc không có missing value và không có dòng trùng. Đây là điểm thuận lợi cho bài toán ML. Tuy nhiên, để đưa dữ liệu qua pipeline Big Data, ta vẫn cần chuẩn hóa tên cột vì các tên như `OccuPational Hazards`, `chronic Lung Disease` có khoảng trắng, chữ hoa/thường không đồng nhất và khó dùng trong CSV, MongoDB field hoặc code Python.

## 6. Chuẩn hóa tên cột

Ta đổi tên cột sang dạng `snake_case` để dùng thống nhất trong HDFS, MapReduce, MongoDB, FastAPI và notebook ML.

In [6]:
column_map = {
    'Patient Id': 'patient_id',
    'Age': 'age',
    'Gender': 'gender',
    'Air Pollution': 'air_pollution',
    'Alcohol use': 'alcohol_use',
    'Dust Allergy': 'dust_allergy',
    'OccuPational Hazards': 'occupational_hazards',
    'Genetic Risk': 'genetic_risk',
    'chronic Lung Disease': 'chronic_lung_disease',
    'Balanced Diet': 'balanced_diet',
    'Obesity': 'obesity',
    'Smoking': 'smoking',
    'Passive Smoker': 'passive_smoker',
    'Chest Pain': 'chest_pain',
    'Coughing of Blood': 'coughing_of_blood',
    'Fatigue': 'fatigue',
    'Weight Loss': 'weight_loss',
    'Shortness of Breath': 'shortness_of_breath',
    'Wheezing': 'wheezing',
    'Swallowing Difficulty': 'swallowing_difficulty',
    'Clubbing of Finger Nails': 'clubbing_of_finger_nails',
    'Frequent Cold': 'frequent_cold',
    'Dry Cough': 'dry_cough',
    'Snoring': 'snoring',
    'Level': 'level',
}

df = df_raw.rename(columns=column_map).copy()
df.columns.tolist()

['patient_id',
 'age',
 'gender',
 'air_pollution',
 'alcohol_use',
 'dust_allergy',
 'occupational_hazards',
 'genetic_risk',
 'chronic_lung_disease',
 'balanced_diet',
 'obesity',
 'smoking',
 'passive_smoker',
 'chest_pain',
 'coughing_of_blood',
 'fatigue',
 'weight_loss',
 'shortness_of_breath',
 'wheezing',
 'swallowing_difficulty',
 'clubbing_of_finger_nails',
 'frequent_cold',
 'dry_cough',
 'snoring',
 'level']

## 7. Làm sạch định danh và nhãn

`patient_id` là khóa logic để lưu collection `patients` trong MongoDB. `level` là nhãn đầu ra nên cần thống nhất đúng 3 giá trị `Low`, `Medium`, `High`.

In [7]:
df['patient_id'] = df['patient_id'].astype(str).str.strip()
df['level'] = df['level'].astype(str).str.strip().str.title()

print('Số Patient Id duy nhất:', df['patient_id'].nunique())
print('Các nhãn Level:', sorted(df['level'].unique()))
display(df['level'].value_counts().rename_axis('level').reset_index(name='count'))

Số Patient Id duy nhất: 1000
Các nhãn Level: ['High', 'Low', 'Medium']


,level,count
0,High,365
1,Medium,332
2,Low,303


## 8. Ép kiểu các đặc trưng số

Theo báo cáo, dataset có 23 đặc trưng số. Ta ép kiểu để tránh lỗi khi train model hoặc khi FastAPI nhận JSON.

In [8]:
feature_cols = [
    'age', 'gender', 'air_pollution', 'alcohol_use', 'dust_allergy',
    'occupational_hazards', 'genetic_risk', 'chronic_lung_disease',
    'balanced_diet', 'obesity', 'smoking', 'passive_smoker', 'chest_pain',
    'coughing_of_blood', 'fatigue', 'weight_loss', 'shortness_of_breath',
    'wheezing', 'swallowing_difficulty', 'clubbing_of_finger_nails',
    'frequent_cold', 'dry_cough', 'snoring'
]
risk_score_cols = [c for c in feature_cols if c not in ['age', 'gender']]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

display(df[feature_cols].dtypes.to_frame('dtype').T)

,age,gender,air_pollution,alcohol_use,dust_allergy,occupational_hazards,genetic_risk,chronic_lung_disease,balanced_diet,obesity,smoking,passive_smoker,chest_pain,coughing_of_blood,fatigue,weight_loss,shortness_of_breath,wheezing,swallowing_difficulty,clubbing_of_finger_nails,frequent_cold,dry_cough,snoring
dtype,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64


## 9. Kiểm định miền giá trị

Phần này rất quan trọng vì dataset y tế đã được mã hóa số:

- `age` nên nằm trong khoảng hợp lý, ở dữ liệu này là 14-73.
- `gender` chỉ có giá trị 1 hoặc 2.
- Các đặc trưng nguy cơ/triệu chứng dùng thang điểm 1-9.
- `level` chỉ gồm `Low`, `Medium`, `High`.

Nếu phát hiện giá trị ngoài miền, cần xem lại file gốc thay vì tự ý sửa vì đây là dữ liệu y tế đã được mã hóa.

In [9]:
quality_checks = [
    ('row_count', len(df), 'Số bản ghi sau khi làm sạch'),
    ('column_count_raw', df.shape[1], 'Số cột sau khi chuẩn hóa tên cột'),
    ('missing_total', int(df.isna().sum().sum()), 'Tổng giá trị thiếu sau khi ép kiểu'),
    ('duplicated_patient_id', int(df['patient_id'].duplicated().sum()), 'Số mã bệnh nhân bị trùng'),
    ('invalid_gender', int((~df['gender'].isin([1, 2])).sum()), 'Gender phải thuộc {1, 2}'),
    ('invalid_age', int((~df['age'].between(0, 120)).sum()), 'Age phải nằm trong khoảng hợp lý'),
    ('invalid_risk_scale', int(sum((~df[col].between(1, 9)).sum() for col in risk_score_cols)), 'Các đặc trưng nguy cơ/triệu chứng phải nằm trong thang 1-9'),
    ('invalid_level', int((~df['level'].isin(['Low', 'Medium', 'High'])).sum()), 'Level phải thuộc Low/Medium/High'),
]

quality_report = pd.DataFrame(quality_checks, columns=['check_name', 'value', 'meaning'])
display(quality_report)

,check_name,value,meaning
0,row_count,1000,Số bản ghi sau khi làm sạch
1,column_count_raw,25,Số cột sau khi chuẩn hóa tên cột
2,missing_total,0,Tổng giá trị thiếu sau khi ép kiểu
3,duplicated_patient_id,0,Số mã bệnh nhân bị trùng
4,invalid_gender,0,"Gender phải thuộc {1, 2}"
5,invalid_age,0,Age phải nằm trong khoảng hợp lý
6,invalid_risk_scale,0,Các đặc trưng nguy cơ/triệu chứng phải nằm tro...
7,invalid_level,0,Level phải thuộc Low/Medium/High


### Nhận xét

Nếu tất cả giá trị kiểm tra đều bằng 0 thì dữ liệu đạt yêu cầu để chuyển sang bước tạo biến phụ trợ. Khác với dataset Telco, ở đây không có cột kiểu `object` cần chuyển như `TotalCharges`; vấn đề chính là chuẩn hóa schema và bảo đảm dữ liệu số đúng miền.

## 10. Tạo nhóm tuổi cho MapReduce/dashboard

Trong báo cáo Big Data, MapReduce thống kê phân bố `Level` theo nhóm tuổi. Vì vậy ta tạo thêm cột `age_group` trước khi xuất CSV.

In [10]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 19, 29, 39, 49, 59, 120],
    labels=['<20', '20-29', '30-39', '40-49', '50-59', '>=60'],
    right=True,
    include_lowest=True,
).astype(str)

display(pd.crosstab(df['age_group'], df['level']))

level,High,Low,Medium
age_group,,,
20-29,85,109,40
30-39,141,79,138
40-49,80,53,74
50-59,19,23,21
<20,19,19,29
>=60,21,20,30


## 11. Mã hóa nhãn cho Machine Learning

Ta mã hóa `Level` theo thứ tự rủi ro: `Low = 0`, `Medium = 1`, `High = 2`. Cách này dễ giải thích hơn `LabelEncoder` mặc định vì `LabelEncoder` có thể sắp xếp nhãn theo alphabet.

In [11]:
level_map = {'Low': 0, 'Medium': 1, 'High': 2}
df['level_encoded'] = df['level'].map(level_map).astype('Int64')

display(df[['level', 'level_encoded']].drop_duplicates().sort_values('level_encoded'))

,level,level_encoded
0,Low,0
1,Medium,1
2,High,2


## 12. Thống kê nhanh giống báo cáo Big Data

Các bảng này dùng để kiểm tra dữ liệu sau khi clean có còn đúng với phần mô tả trong báo cáo hay không.

In [12]:
level_distribution = df['level'].value_counts().rename_axis('level').reset_index(name='count')
level_distribution['ratio'] = level_distribution['count'] / len(df)

smoking_level = pd.crosstab(df['smoking'], df['level'])
age_level = pd.crosstab(df['age_group'], df['level'])

display(level_distribution)
display(age_level)
display(smoking_level)

,level,count,ratio
0,High,365,0.365
1,Medium,332,0.332
2,Low,303,0.303


level,High,Low,Medium
age_group,,,
20-29,85,109,40
30-39,141,79,138
40-49,80,53,74
50-59,19,23,21
<20,19,19,29
>=60,21,20,30


level,High,Low,Medium
smoking,,,
1,0,61,120
2,70,81,71
3,0,71,101
4,19,40,0
5,0,0,10
6,10,30,20
7,187,20,0
8,79,0,10


## 13. Tạo dữ liệu đầu ra

Ta tạo 2 phiên bản dữ liệu sạch:

- `cancer_patients_clean_hdfs.csv`: dùng cho HDFS/MapReduce/MongoDB, giữ `level` dạng chữ để dashboard dễ đọc.
- `cancer_patients_ml_ready.csv`: dùng cho huấn luyện ML, có thêm `level_encoded` để làm nhãn số.

In [13]:
OUTPUT_DIR = Path(r'C:\\Users\\dzyuu\\Documents\\Codex\\2026-06-13\\files-mentioned-by-the-user-cancer\\outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

hdfs_columns = ['patient_id'] + feature_cols + ['age_group', 'level']
ml_columns = ['patient_id'] + feature_cols + ['level_encoded', 'level']

df_hdfs = df[hdfs_columns].drop_duplicates(subset=['patient_id'], keep='first').reset_index(drop=True)
df_ml = df[ml_columns].drop_duplicates(subset=['patient_id'], keep='first').reset_index(drop=True)

hdfs_path = OUTPUT_DIR / 'cancer_patients_clean_hdfs.csv'
ml_path = OUTPUT_DIR / 'cancer_patients_ml_ready.csv'
quality_path = OUTPUT_DIR / 'cancer_cleaning_quality_report.csv'

df_hdfs.to_csv(hdfs_path, index=False, encoding='utf-8-sig')
df_ml.to_csv(ml_path, index=False, encoding='utf-8-sig')
quality_report.to_csv(quality_path, index=False, encoding='utf-8-sig')

print('Đã xuất:', hdfs_path)
print('Đã xuất:', ml_path)
print('Đã xuất:', quality_path)

Đã xuất: C:\Users\dzyuu\Documents\Codex\2026-06-13\files-mentioned-by-the-user-cancer\outputs\cancer_patients_clean_hdfs.csv
Đã xuất: C:\Users\dzyuu\Documents\Codex\2026-06-13\files-mentioned-by-the-user-cancer\outputs\cancer_patients_ml_ready.csv
Đã xuất: C:\Users\dzyuu\Documents\Codex\2026-06-13\files-mentioned-by-the-user-cancer\outputs\cancer_cleaning_quality_report.csv


## 14. Kết luận xử lý dữ liệu

Sau khi clean, dataset đã sẵn sàng cho pipeline của báo cáo:

1. **Excel -> CSV:** dữ liệu được chuẩn hóa tên cột và xuất ra CSV.
2. **CSV -> HDFS/MapReduce:** file `cancer_patients_clean_hdfs.csv` có `age_group`, `smoking`, `level` để thống kê batch.
3. **MongoDB:** `patient_id` dùng làm định danh, các field dạng `snake_case` phù hợp với document JSON.
4. **Python ML:** file `cancer_patients_ml_ready.csv` có 23 đặc trưng số và nhãn `level_encoded` cho Random Forest/XGBoost.
5. **FastAPI/WinForm:** schema đã ổn định nên có thể dùng cùng tên field khi gửi JSON dự đoán.